# Comparaison reproductible du risque de diab?te
Aucun r?sultat pr?rempli : ex?cuter avec un CSV r?el document?. Les artefacts historiques restent inchang?s. Ce notebook d?l?gue le protocole au module versionn? `training.py`.


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name != 'ml_api':
    ROOT = ROOT / 'ml_api'
sys.path.insert(0, str(ROOT.resolve()))
from training import run_experiment
CSV_PATH = Path(os.environ.get('ML_DATA_PATH', str(ROOT / 'data' / 'diabetes_binary.csv')))
OUTPUT = Path(os.environ.get('ML_RUN_DIR', str(ROOT / 'runs' / 'experiment_42')))
TARGET = os.environ.get('ML_TARGET', 'Diabetes_binary')
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Fournir un CSV r?el : {CSV_PATH}. Voir README.md.')


## Protocole
Cible explicitement binaire {0,1}. Les valeurs manquantes et encodages invalides sont rejet?s. Les doublons exacts sont retir?s ; les vecteurs identiques restent dans le m?me groupe afin de ne pas traverser les partitions. Un pli sur cinq est r?serv? au test. Sur les autres plis, validation crois?e ? cinq plis ; scaler ajust? dans chaque pli. S?lection par average precision moyenne en validation, jamais par score du test. Hyperparam?tres fixes, seed 42, CPU et un thread. Le seuil de classification est 0,5.


In [ ]:
cv_summary, test_metrics, manifest = run_experiment(CSV_PATH, OUTPUT, TARGET)
display(cv_summary)
display(test_metrics)
print('Mod?le s?lectionn? en validation :', manifest['selected_by_training_cv_average_precision'])


## ?valuation et interpr?tation
ROC-AUC, average precision, accuracy, balanced accuracy, pr?cision, rappel, F1, Brier et log-loss ; rapports par classe, matrices de confusion, ROC, pr?cision-rappel et calibration. La moyenne et l??cart-type des plis d?crivent la variabilit? de validation, pas un intervalle de confiance. Les m?triques de test concernent les nouveaux mod?les seulement : la provenance des anciens artefacts ne permet pas de garantir un test ind?pendant.


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'evaluation.png')))
display(Image(filename=str(OUTPUT / 'shap_importance.png')))


## SHAP r?els
Contributions TreeSHAP exactes natives XGBoost (`pred_contribs=True`, `approx_contribs=False`), sur au plus 500 lignes du test pour l?analyse globale. La somme des contributions et de la base reconstruit la marge ; la sigmo?de donne la probabilit?. L?unit? est le log-odds : une contribution n?est ni un pourcentage de risque ni une preuve causale.

## Reproductibilit? et limites
Le dossier de sortie contient mod?les, pr?dictions, m?triques, figures, indices des partitions, SHA-256 du CSV, param?tres et versions install?es. Un dossier existant est refus?. Consigner aussi source/licence, population, ann?e, d?finition de la cible et ?ventuel ?quilibrage du CSV. Des donn?es ?quilibr?es modifient la pr?valence et l?interpr?tation des probabilit?s. Aucune validation clinique ou externe n?est revendiqu?e. Voir README.md avant toute utilisation des r?sultats dans le m?moire.
